# File 01/1 – Download data

### DESCRIPTION:
 This script downloads the primary source files (Old East Slavic, (Old) Church Slavonic) from the TOROT Treebank GitHub page, along with the Modern Russian and English translations from the PROEIL Syntacticus site. 
 If the files already exist, they will be overwritten! 


#### INPUT FILES:
- Treebank Releases (Github)
    - url: https://github.com/torottreebank/treebank-releases/archive/refs/tags/20180919.zip
- Translation Files (Github)
    - orv: https://raw.githubusercontent.com/proiel/syntacticus-dictionaries/master/orv.xml
    - chu: https://raw.githubusercontent.com/proiel/syntacticus-dictionaries/master/chu.xml
      
#### OUTPUT FILES:
- ../source_data/treebank-releases-20180919/*xml
- ../source_data/translations/orv.xml
- ../source_data/translations/chu.xml

## Imports 

In [ ]:
# standard library imports
import os
import glob
import shutil
import zipfile
import tempfile
from pathlib import Path

# third-party imports
import requests

## Overwrite Behavior

- If `FORCE_OVERWRITE = False` (default):
  - If target directory exists and contains XML files → skip download
  - Otherwise → create directory and download files
  - The same is true for the translation files<br></br>

- If `FORCE_OVERWRITE = True`:
  - Delete the target directories (if they exist)
  - Recreate the directories for primary sources and translation files
  - Download all files anew

### Warning
If you change the download URL (e.g. to another release), you must set `FORCE_OVERWRITE = True` to avoid using outdated data.

In [ ]:
FORCE_OVERWRITE = False

## Download primary source files 
(i.e. treebank files, VERSION: 20180919) 
- skips all files which do not end in xml

In [ ]:
# --- Config ---
url = "https://github.com/torottreebank/treebank-releases/archive/refs/tags/20180919.zip"
zip_name = "treebank_20180919.zip"
target_dir = "../source_data/treebank-releases-20180919"

# --- Check / Prepare target dir ---
if os.path.exists(target_dir) and os.listdir(target_dir):
    if not FORCE_OVERWRITE:
        print("Primary source files already exist -> SKIPPED DOWNLOAD")
    shutil.rmtree(target_dir)

os.makedirs(target_dir, exist_ok=True)

# --- 1) Download ZIP ---
r = requests.get(url, stream=True, timeout=30)
r.raise_for_status()

with open(zip_name, "wb") as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)

# --- 2) Unzip into temp directory ---
tmp_dir = "__tmp_treebank_20180919"
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(tmp_dir)

# --- 3) Move only XML files ---
extracted_root = os.path.join(tmp_dir, os.listdir(tmp_dir)[0])

for item in os.listdir(extracted_root):
    if not item.endswith(".xml"):
        continue

    src = os.path.join(extracted_root, item)
    dst = os.path.join(target_dir, item)

    shutil.move(src, dst)

# --- 4) Cleanup ---
os.remove(zip_name)
shutil.rmtree(tmp_dir)

# --- Assertions ---
assert not os.path.exists(tmp_dir)
assert os.path.exists(target_dir)

xml_files = glob.glob(os.path.join(target_dir, "*.xml"))
assert len(xml_files) > 0, "No XML files found"

### Assertions concerning the primary source files:
- temp directory removed after extraction  
- target directory created  
- XML files successfully extracted (non-empty)

In [ ]:
## temporary directory was deleted after unzipping
assert not os.path.exists(tmp_dir)
## directory containing the treebank-releases XML input files was created
assert os.path.exists(target_dir)
## treebank-releases dir contains the downloaded XML files
xml_files = glob.glob(os.path.join(target_dir, "*.xml"))
assert xml_files, "No XML files found"

## Translation files:
- orv.xml (English and Russian translations for Old Russian texts in XML format)
- chu.xml (English and Russian translations for Old Church Slavonic texts in XML format)

In [ ]:
# target directory
translation_dir = "../source_data/translations"
if FORCE_OVERWRITE:
    shutil.rmtree(translation_dir, ignore_errors=True)

urls = {
    "orv": "https://raw.githubusercontent.com/proiel/syntacticus-dictionaries/master/orv.xml",
    "chu": "https://raw.githubusercontent.com/proiel/syntacticus-dictionaries/master/chu.xml",
}

# create expected file names from the urls 
expected_files = {f"{name}.xml" for name in urls}


os.makedirs(translation_dir, exist_ok=True)

# save potential paths for the xml files from the url dict downloads
existing_files = {
    os.path.basename(p) 
    for p in glob.glob(os.path.join(translation_dir, "*.xml"))
}

missing_files = expected_files - existing_files
redundant_files = existing_files - expected_files

# remove redundant files
for f in redundant_files:
    os.remove(os.path.join(translation_dir, f))

# skip download if expected amount of files exists 
# in translation_dir 
if not missing_files and not FORCE_OVERWRITE:
    print("All translation files exist → SKIPPED DOWNLOAD")
else:
    for name, url in urls.items():
        filename = f"{name}.xml"
        if filename not in missing_files and not FORCE_OVERWRITE:
            continue

        target_path = os.path.join(translation_dir, filename)

        response = requests.get(url, timeout=30)
        response.raise_for_status()

        with open(target_path, "wb") as f:
            f.write(response.content)


### Assertions concerning the translation files

In [ ]:
# assert that the target directory exists
assert os.path.exists(translation_dir)

# collect XML files in the target directory
xml_files = glob.glob(os.path.join(translation_dir, "*.xml"))

# assert that XML files were downloaded successfully
assert xml_files, "No XML translation files found"

# assert that all expected XML files are present
assert len(xml_files) == len(urls), "Missing XML translation files"